In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import datetime, time
import os
from dotenv import load_dotenv


import ast
from itertools import chain

import matplotlib.pyplot as plt

import re
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [2]:

def establish_db_connection(server, database, username, password, driver):
    connection_string = (
        f"mssql+pyodbc://{username}:{password}@{server}/{database}"
        f"?driver={driver.replace(' ', '+')}"
    )
    engine = create_engine(connection_string)

    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT @@VERSION"))
            for row in result:
                print("Connected successfully. SQL Server version:")
                print(row[0])
            return engine
    except Exception as e:
        print("Connection failed:")
        print(e)
        return None

# Adjust to include error handling for the db connection method



In [3]:
def load_env():
    load_dotenv(dotenv_path="creds\\.env")


def SERVER_conn(input_site):

    load_env()

    # DB server
    site_server = os.getenv(input_site)
    
    
    paramz = {
        "site": os.getenv('site_server'),
        "userName": os.getenv('USER_NAME'),
        "Password": os.getenv('PASSWORD_dev-test'),
        "Driver": os.getenv("ODBC_DRIVER")
    }

    db = os.getenv(input_site)

    server_conn = establish_db_connection(
        paramz["site"],
        db, 
        paramz["userName"],     
        paramz["Password"],
        paramz["Driver"])
        
    return server_conn


def db_request(query, server_conn_str):
    if server_conn_str is None:
        raise Exception("Database connection failed. Please check your credentials and connection settings.")

    # start_time = time.time()
    df = pd.read_sql(query, server_conn_str)

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"db_request must return a DataFrame, got {type(df)}")

    # end_time = time.time()
    # print(f"Query executed in {end_time - start_time:.2f} seconds")
    return df


In [ ]:

# RO_all = "SELECT *  from Ops_tblRepairOrder where fldLastUpdated > '2020-01-1' AND fldStatus = 3 AND fldDivision IN (1)"
# query_all_requests = "SELECT *  from Ops_tblRequests where fldLastUpdated > '2020-01-1' AND fldAddWorkStatus IN (100, 300, 400)" 
# query_all_LabourLine = "SELECT *  from Ops_tblLabourLine where fldLastUpdated > '2020-01-1'"
# query_all_PartsLine = "SELECT *  from Ops_tblPartsLine where fldLastUpdated > '2020-01-1'"

# # More queries
# 
# 
# 

# get all F150 closed RO with relevant requests
RO_all = "SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_requests = "SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
query_all_PartsLine = "SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"
 

In [5]:
# search for op_codes_based_on_key_words

# select * from 
# Ops_tblOpCode2
# where fldDescription like ('%Water Pump%')

In [6]:


def pull_data_by_server(server_conn_str):
    # pull data for 
    RO_tbl = db_request(RO_all, server_conn_str)
    request_tbl = db_request(query_all_requests, server_conn_str)
    labourline_tbl = db_request(query_all_LabourLine, server_conn_str)
    partslines_tbl = db_request(query_all_PartsLine, server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl

def pull_data_by_server_with_args(server_conn_str, queries, modelName):
    # pull data for 

    # RO_tbl = server_conn_str.execute(queries["RO_tbl"], {"model": modelName}).fetchall()
    # request_tbl = server_conn_str.execute(queries["Req_tbl"], {"model": modelName}).fetchall()
    # labourline_tbl = server_conn_str.execute(queries["Labour_tbl"], {"model": modelName}).fetchall()
    # partslines_tbl = server_conn_str.execute(queries["Parts_tbl"], {"model": modelName}).fetchall()

    RO_tbl = db_request(queries["RO_tbl"], server_conn_str)
    request_tbl = db_request(queries["Req_tbl"], server_conn_str)
    labourline_tbl = db_request(queries["Labour_tbl"], server_conn_str)
    partslines_tbl = db_request(queries["Parts_tbl"], server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl

 

In [7]:

# # pull data for 
# RO_tbl_vw_174 = db_request(RO_all, vw_18_db)
# request_tbl_vw_174 = db_request(query_all_requests, vw_18_db)
# labourline_tbl_vw_174 = db_request(query_all_LabourLine, vw_18_db)
# partslines_tbl_vw_174 = db_request(query_all_PartsLine, vw_18_db)


In [8]:

# function to drop empty columns
def drop_empty_columns(df):
    df_cleaning = df.copy()
    # drop empty columns - must all empty
    df_cleaning = df_cleaning.dropna(axis=1, how='all')
    
    return df_cleaning
 

In [9]:
# function to filter columns
def filter_for_essential_columns(df, essential_cols):
    df_selected = df[essential_cols].copy()
    return df_selected

In [10]:
# Defined essential columns for each table

essential_columns_request_tbl = ['fldId', 'fldWorkItemRef', 'fldSequence', 'fldDescription',
       'fldRequestCodeRef', 'fldRequestCode', 'fldRequestedTime', 'fldOrderNumber',
        'fldLastUpdated']


essential_cols_labourline_tbl = ['fldID', 'fldRequestRef', 'fldOpCodeRef',
       'fldActualHours', 'fldSoldHours', 'fldDescription',
       'fldAddedDate']

essential_cols_partlines_tbl = ['fldID', 'fldRequestRef', 'fldSequence', 'fldPartNumber', 'fldPartDesc',
       'fldRequested', 'fldShipped', 'fldOrderType', 'fldDateAdded', 'PC_PartDesc', 'fldPartsMasterRef']


essential_cols_RO_tbl = ['fldId', 'fldContactRef', 'fldVehicleRef', 'fldDateOpened',
       'fldDateClosed'
       ]
       
  

In [11]:

def clean_datset(df, tbl_type):
    df_dropped_empty_cols = drop_empty_columns(df)

    if tbl_type == "request":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_columns_request_tbl)
    
    elif tbl_type == "labourline":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_labourline_tbl)

    elif tbl_type == "partslines":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_partlines_tbl)

    elif tbl_type == "RO_tbl":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_RO_tbl)
        # remove
        
    return df_filtered
 

In [12]:

# request_tbl_vw_174 = clean_datset(request_tbl_vw_174, tbl_type="request")
# labor_tbl_vw_174 = clean_datset(labourline_tbl_vw_174, tbl_type="labourline")
# parts_tbl_vw_174 = clean_datset(partslines_tbl_vw_174, tbl_type="partslines")
# RO_tbl_vw_174 = clean_datset(RO_tbl_vw_174, tbl_type="RO_tbl")

#### Find a list of labour and parts for the following repair jobs 

- water pump 
- Timing belt
- Electrical - exterior lights


In [144]:
def search_columns_for_keyword(df, keyword, column):
    if (column not in df.columns) or column=="":
        raise ValueError(f"Column '{column}' does not exist in the DataFrame.")
    filtered_df = df[df[column].str.contains(keyword, case=False, na=False)]
    return filtered_df

def get_top_ten_opcodes(df):
    top_ten = df["fldRequestCode"].value_counts().head(20)
    return top_ten

def search_request_by_opcode(df, opcode):
    search_result = df[df["fldRequestCode"]== opcode]
    
    return search_result

def search_request_by_list_of_opcodes(df, opcode_list):
    search_result = df[df["fldRequestCode"].isin(opcode_list)]
    
    return search_result



def part_items_metrics(parts_df):

    uniq_item_by_description = set(parts_df['fldPartDesc'].unique())
    metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])

    for desc in uniq_item_by_description:
        item_count = len(parts_df[parts_df['fldPartDesc'] == desc]) 
        total_units = parts_df[parts_df['fldPartDesc'] == desc]['fldRequested'].sum()
        uniq_partNumbers = parts_df[parts_df['fldPartDesc'] == desc]['fldPartNumber'].unique().tolist()
        new_row = {
                    'partDesc': desc, 
                    '#UniqParts': item_count, 
                    '#Qty': total_units,
                    'uniq_partNumbers': uniq_partNumbers
                    }
        
        # metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])
        
        new_row_df = pd.DataFrame([new_row]).reindex(columns=metrics.columns)
        metrics = pd.concat([metrics, new_row_df], ignore_index=True)
    return metrics




# def parts_summary(parts_tbl_df, total_req_count):
#     # Count occurrences of each unique part
#     # part_counts = parts_tbl_df['fldPartDesc'].value_counts()

#     part_counts = parts_tbl_df.groupby("fldPartDesc", as_index=False).agg(
#     count = ("fldPartNumber", "count"),
#     PartNum = ("fldPartNumber", lambda x: list(x.unique()))
#     ).sort_values("count", ascending=False)
#     part_counts = part_counts[~part_counts["fldPartDesc"].str.contains('ENV Fee|Core charge', case=False, regex=True)]


#     # Calculate percentage occurrence
#     part_counts["perc_occurence"] = round((part_counts['count'] / total_req_count * 100), 2)
    

#     # Sort for readability
#     metrics = metrics.sort_values(by='#perc_occurence', ascending=False)


#     # display(metrics)
#     return metrics



def parts_summary_v1(parts_tbl_df, total_req_count, similarity_threshold, ignore_words):
    """
    Summarizes parts occurrence and groups similar descriptions based on keyword similarity.
    
    Parameters:
    ----------
    parts_tbl_df : pd.DataFrame
        DataFrame containing part descriptions and request references.
    total_req_count : int
        Total number of requests for percentage calculation.
    similarity_threshold : float, optional (default=0.2)
        Jaccard similarity threshold for grouping descriptions.
    ignore_words : list of str, optional
        Words to ignore when determining similarity and forming combined names.
    
    Returns:
    -------
    pd.DataFrame
        DataFrame with combined part names and % occurrence.
    """
    
    if ignore_words is None:
        ignore_words = []
    
    # Normalize descriptions
    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    # Calculate initial metrics
    metrics = (
        parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
        .drop_duplicates()
        .groupby('fldPartDesc')
        .size()
        .reset_index(name='Count')
    )

    # Tokenize descriptions and remove ignored words
    metrics['Tokens'] = metrics['fldPartDesc'].apply(
        lambda x: set(word for word in re.split(r'\W+', x) if word and word not in [w.upper() for w in ignore_words])
    )

    # Group similar descriptions
    grouped = []
    visited = set()

    for i, row_i in metrics.iterrows():
        if i in visited:
            continue
        group = [i]
        for j, row_j in metrics.iterrows():
            if j in visited or i == j:
                continue
            # Jaccard similarity
            sim = len(row_i['Tokens'] & row_j['Tokens']) / len(row_i['Tokens'] | row_j['Tokens'])
            if sim >= similarity_threshold:
                group.append(j)
        visited.update(group)
        grouped.append(group)

    # Aggregate groups
    new_rows = []
    for group in grouped:
        part_names = metrics.loc[group, 'fldPartDesc'].tolist()
        counts = metrics.loc[group, 'Count'].sum()
        common_tokens = set.intersection(*metrics.loc[group, 'Tokens']) if len(group) > 1 else metrics.loc[group, 'Tokens'].iloc[0]
        common_name = " ".join(sorted(common_tokens)) if common_tokens else part_names[0]
        new_rows.append({'Part': common_name, 'Count': counts})

    # Create final DataFrame
    final_df = pd.DataFrame(new_rows)
    final_df['%Occurrence'] = (final_df['Count'] / total_req_count) * 100
    final_df['%Occurrence'] = final_df['%Occurrence'].round(2)
    final_df = final_df.sort_values(by='%Occurrence', ascending=False).reset_index(drop=True)
    final_df = final_df[["Part", "%Occurrence"]]
    return final_df



def parts_summary(parts_tbl_df):
    
    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    # Filter out non-relevant items based on keywords
    parts_tbl_df = parts_tbl_df[~parts_tbl_df["fldPartDesc"].str.contains("ENV FEE|CORE|Fluids", case=False, regex = True)]

    # count unique request references for each part description
    total_req_count = len(parts_tbl_df['fldRequestRef'].unique())



    # Agregate unique request counts by part description, count unique request references for each part description
    metrics = (parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
               .drop_duplicates()
               .groupby('fldPartDesc', as_index=False)
                .agg(
                    uniq_fldPartDesc_count= ("fldRequestRef","nunique")
                    )
                )
        
    # Calculate percentage occurrence for each part description
    metrics["freq_perc"] = (metrics["uniq_fldPartDesc_count"]/total_req_count * 100).round(2)

    # Create a list of unique part numbers for each part description
    partNumber_uniqueList = (parts_tbl_df.groupby('fldPartDesc',as_index=False)
                                .agg(PartNumbers = ('fldPartNumber', lambda x: list(pd.unique(x))))
                            )

    # Merge metrics with part numbers and sort by frequency percentage
    results = metrics.merge(partNumber_uniqueList, on = 'fldPartDesc')
    results = results.sort_values("freq_perc", ascending=False)

    #  Reset index and rename columns for better readability, also set index to start from 1 instead of 0
    results = results.reset_index(drop=True).set_axis(range(1, len(results) + 1))

    # # print sample size and unique opcodes for reference
    # print(f"Sample size: {total_req_count} ROs")
    
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    results.columns =["Part","Count", "frequency_%", "PartNumbers"]

    return results[["Part", "frequency_%", "PartNumbers"]]



def search_parts_and_labour_by_req_id(labor, parts, req_id):

    if "fldRequestRef" not in labor.columns or "fldRequestRef" not in parts.columns:
        raise ValueError("The required column 'fldRequestRef' does not exist in one of the DataFrames.")
    
    labor_result = labor[labor["fldRequestRef"]== req_id]
    parts_result = parts[parts["fldRequestRef"]== req_id]
        
    print(f"Labour items for Request ID {req_id}:")
    display(labor_result)
    print(f"Parts items for Request ID {req_id}:")
    display(part_items_metrics(parts_result))
    

In [14]:

def remove_invalid_opcodes(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(df)}")

    required_cols = ['fldFlatHours', 'fldTimeAllowed', 'fldCode']

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    # Apply filter
    return df[(df["fldFlatHours"] > 0) & (df["fldTimeAllowed"] > 0)]
    # return df



def get_valid_op_codes_by_keyword(db_conn, key_word: str) -> list:
    query = f"SELECT * FROM Ops_tblOpCode2 WHERE fldDescription LIKE '%{key_word}%'"
    search_result = db_request(query, db_conn)

    if search_result.empty:
        raise LookupError(f"No matching data for query: {query}")

    filter_results = remove_invalid_opcodes(search_result)

    if filter_results.empty:
        raise LookupError(f"No matching records found for keyword: {key_word}")

    return filter_results



def get_all_op_codes(db_conn) -> pd.DataFrame:
    """
    Retrieves all opcodes from the database.
    """
    query = "SELECT * FROM Ops_tblOpCode2"
    
    search_result = db_request(query, db_conn)

    return search_result


In [15]:

def clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl):
    RO_tbl_cleaned = clean_datset(RO_tbl, tbl_type="RO_tbl")
    request_tbl_cleaned = clean_datset(request_tbl, tbl_type="request")
    labourline_tbl_cleaned = clean_datset(labourline_tbl, tbl_type="labourline")
    partslines_tbl_cleaned = clean_datset(partslines_tbl, tbl_type="partslines")

    return RO_tbl_cleaned, request_tbl_cleaned, labourline_tbl_cleaned, partslines_tbl_cleaned


In [16]:

# Function to count the number of times a unique part item appears on a repair job

def parts_analysis(part_items, tracker_count_part_item_once_per_job, parts_summary_df):
    for index, row in part_items.iterrows():
            part_number = row["fldPartNumber"]
            part_desc = row["fldPartDesc"]

            # Count occurrence of each part item used on job             
            if (part_number in parts_summary_df["Part Number"].values) and (part_number not in tracker_count_part_item_once_per_job):
                parts_summary_df.loc[parts_summary_df["Part Number"] == part_number, "Occurrence_count"] += 1
                tracker_count_part_item_once_per_job.add(part_number)
            else:
                new_row = {
                    "Part Number": part_number,
                    "Part Description": part_desc,
                    "Occurrence_count": 1
                }
                parts_summary_df = pd.concat([parts_summary_df, pd.DataFrame([new_row])], ignore_index=True) 
                tracker_count_part_item_once_per_job.add(part_number)
                parts_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)


    return parts_summary_df, tracker_count_part_item_once_per_job
 

In [17]:


def filter_rows_by_keywords(df, column_name, keywords=[[], []], return_print=True):
    """
    Filters rows in a DataFrame where:
    - All keywords in the first sub-array must be present (AND logic).
    - At least one keyword in the second sub-array must be present (OR logic).
    
    Parameters:
    ----------
    df : pd.DataFrame
        The DataFrame to search.
    column_name : str
        The name of the column to search within.
    keywords : list of two lists
        keywords[0] = list of must-have keywords (AND condition)
        keywords[1] = list of optional keywords (at least one required)
    return_counts : bool, optional (default=True)
        If True, returns value counts of the filtered column.
        If False, returns the filtered DataFrame.
    
    Returns:
    -------
    pd.Series or pd.DataFrame
        Value counts of the filtered column or the filtered DataFrame.
    """
    
    must_have = keywords[0]
    optional = keywords[1]
    
    # Build regex for must-have keywords (AND logic using lookaheads)
    must_pattern = "".join(f"(?=.*{re.escape(word)})" for word in must_have)
    
    # Build regex for optional keywords (OR logic using |)
    optional_pattern = "|".join(re.escape(word) for word in optional)
    
    # Combine patterns: must-have AND (optional OR empty if none)
    if optional:
        pattern = f"{must_pattern}(?=.*(?:{optional_pattern}))"
    else:
        pattern = must_pattern
    
    # Apply filter
    mask = df[column_name].str.contains(pattern, case=False, regex=True, na=False)
    filtered_df = df[mask]

    # print(f"Sample size: {len(filtered_df)}")
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    key_columns= ['fldRequestCode', 'fldDescription']

    
    return filtered_df[key_columns] if return_print else filtered_df


In [18]:
def labour_items_analysis(labour_items, tracker_count_labour_item_once_par_job, labour_summary_df):
    for index, row in labour_items.iterrows():
            op_code = row["fldOpCodeRef"]
            labour_desc = row["fldDescription"]

            if op_code in labour_summary_df["fldOpCodeRef"].values and op_code not in tracker_count_labour_item_once_par_job:
                labour_summary_df.loc[labour_summary_df["fldOpCodeRef"] == op_code, "Occurrence_count"] += 1
                tracker_count_labour_item_once_par_job.add(op_code)
            else:
                new_row = {
                    "fldOpCodeRef": op_code,
                    "fldDescription": labour_desc,
                    "Occurrence_count": 1
                }
                labour_summary_df = pd.concat([labour_summary_df, pd.DataFrame([new_row])] , ignore_index=True )
                tracker_count_labour_item_once_par_job.add(op_code)
                labour_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)  
    return labour_summary_df, tracker_count_labour_item_once_par_job

In [19]:
def requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, request_summary_df):
    
    new_row = {
            "Request ID": req_id,
            "Description": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"].values[0],
            "fldRequestCode": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldRequestCode"].values[0],
            "#_PartItems": len(part_items),
            "#_LaborItems": len(labour_items)
            }

    request_summary_df = pd.concat([request_summary_df, pd.DataFrame([new_row])], ignore_index=True )
    request_summary_df.sort_values(by="#_PartItems", ascending=False, inplace=True)
    
    return request_summary_df

In [20]:


def run_analysis(labourline_tbl, partslines_tbl, request_tbl):


    filtered_requests_df = request_tbl
    
    # search from labour line and part line where fldRequestRef in filtered_requests_df['fldId']
    filtered_labour_df = labourline_tbl[labourline_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]
    filtered_parts_df = partslines_tbl[partslines_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]


    parts_summary_df = pd.DataFrame(columns=["Part Number", "Part Description", "Occurrence_count"])
    labour_summary_df = pd.DataFrame(columns=["fldOpCodeRef", "fldDescription", "Occurrence_count"])
    requestLine_summary_df = pd.DataFrame(columns=["Request ID", "Description","fldRequestCode", "#_PartItems", "#_LaborItems"])
 
    for items in filtered_requests_df['fldId'].values:
        req_id = items

        tracker_count_part_item_once_per_job = set()
        tracker_count_labour_item_once_par_job = set()
        

    # print(filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"])

        # search for all parts and labour lines for this req_id
        part_items = filtered_parts_df[filtered_parts_df["fldRequestRef"]== req_id]
        labour_items = filtered_labour_df[filtered_labour_df["fldRequestRef"]==req_id]

        requestLine_summary_df = requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, requestLine_summary_df)

        if not part_items.empty:
            parts_summary_df, tracker_count_part_item_once_per_job = parts_analysis(
                                                                                    part_items=part_items, 
                                                                                    tracker_count_part_item_once_per_job = tracker_count_part_item_once_per_job, 
                                                                                    parts_summary_df = parts_summary_df
                                                                                    )


        # if not labour_items.empty:
        #     labour_summary_df, tracker_count_labour_item_once_par_job = labour_items_analysis(
        #                                                                                 labour_items=labour_items, 
        #                                                                                 tracker_count_labour_item_once_par_job=tracker_count_labour_item_once_par_job, 
        #                                                                                 labour_summary_df=labour_summary_df
        #                                                                                 ) 
                                                           

    return filtered_parts_df

In [21]:
def plot_stats(requestLine_summary_df):

    parts_stats =  (
    requestLine_summary_df["#_PartItems"]
    .value_counts()
    .reset_index()
    .rename(columns={'index': '#_PartItems', '#_PartItems': '#Parts'})
    )

    labour_stats =  (
        requestLine_summary_df["#_LaborItems"]
        .value_counts()
        .reset_index()
        .rename(columns={'index': '#_LaborItems', '#_LaborItems': '#labour'})
    )

    # Sort for better visualization
    parts_stats = parts_stats.sort_values(by='#Parts').reset_index(drop=True)
    labour_stats = labour_stats.sort_values(by='#labour').reset_index(drop=True)


    # Compute stats for Parts
    mean_parts = parts_stats['#Parts'].mean()
    median_parts = parts_stats['#Parts'].median()
    mode_parts = parts_stats['#Parts'].mode()[0]

    # Compute stats for Labour
    mean_labour = labour_stats['#labour'].mean()
    median_labour = labour_stats['#labour'].median()
    mode_labour = labour_stats['#labour'].mode()[0]


    # Plot Parts line
    plt.plot(parts_stats['#Parts'], parts_stats['count'], color='blue', marker='o', label='Parts')

    # Plot Labour line
    plt.plot(labour_stats['#labour'], labour_stats['count'], color='green', marker='o', label='Labour')


    # Add reference lines for mean
    plt.axvline(mean_parts, color='blue', linestyle='--', alpha=0.5, label=f'Parts Mean: {mean_parts:.2f}')
    plt.axvline(mean_labour, color='green', linestyle='--', alpha=0.5, label=f'Labour Mean: {mean_labour:.2f}')


    # Annotate median and mode
    plt.text(parts_stats['#Parts'].max(), median_parts, f'Median: {median_parts}', color='blue')
    plt.text(parts_stats['#Parts'].max(), mode_parts, f'Mode: {mode_parts}', color='blue')
    plt.text(labour_stats['#labour'].max(), median_labour, f'Median: {median_labour}', color='green')
    plt.text(labour_stats['#labour'].max(), mode_labour, f'Mode: {mode_labour}', color='green')


    # Labels and title
    plt.xlabel('Item Count')
    plt.ylabel('Frequency')
    plt.title('Parts vs Labour Items with Summary Stats')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:


def _normalize_partnumbers_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return [str(v).strip() for v in parsed if str(v).strip()]
            except Exception:
                pass
        return [p.strip() for p in s.split(",") if p.strip()]
    return [str(x).strip()] if str(x).strip() else []

def _dedupe_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _glob_to_regex(token: str, match_mode: str = "contains") -> str:
    """
    Convert a keyword with glob wildcards to regex.
      * => .*
      ? => .
    We escape everything else.
    """
    t = str(token).strip()
    if not t:
        return ""

    esc = re.escape(t)
    esc = esc.replace(r"\*", ".*").replace(r"\?", ".")
    if match_mode == "word":
        return rf"\b{esc}\b"
    return esc

def combine_parts_by_keyword_groups(
    df: pd.DataFrame,
    groups: dict,
    part_col: str = "Part",
    freq_col: str = "frequency_%",
    partnums_col: str = "PartNumbers",
    match_mode: str = "contains",  # "contains" or "word"
    dedupe_partnums: bool = True,
    combined_parts_col: str = "CombinedParts",
    matched_keywords_col: str = "MatchedKeywords",
) -> pd.DataFrame:
    """
    groups format:
      {
        "CANONICAL": [[include_any_patterns], [exclude_any_patterns]],
        ...
      }

    include_any_patterns: OR logic (at least one must match)
    exclude_any_patterns: NOT logic (none may match)

    Matching is case-insensitive (case=False) per your requirement.
    """

    required = {part_col, freq_col, partnums_col}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    if match_mode not in {"contains", "word"}:
        raise ValueError("match_mode must be 'contains' or 'word'")

    work = df.copy()
    work[freq_col] = pd.to_numeric(work[freq_col], errors="coerce").fillna(0)
    work[partnums_col] = work[partnums_col].apply(_normalize_partnumbers_cell)

    part_series = work[part_col].fillna("").astype(str)

    unassigned = pd.Series(True, index=work.index)
    matched_rows_out = []

    for canonical_name, rule in groups.items():
        if not isinstance(rule, (list, tuple)) or len(rule) != 2:
            raise ValueError(f"Group '{canonical_name}' must be [[include_any],[exclude_any]]")

        include_any, exclude_any = rule
        include_any = include_any or []
        exclude_any = exclude_any or []

        if len(include_any) == 0:
            continue

        # Build OR regex for includes
        include_patterns = [_glob_to_regex(k, match_mode) for k in include_any if str(k).strip()]
        include_patterns = [p for p in include_patterns if p]
        include_or = "(?:" + "|".join(include_patterns) + ")"

        include_mask = part_series.str.contains(include_or, case=False, regex=True, na=False)

        # Build OR regex for excludes (if any)
        if exclude_any:
            exclude_patterns = [_glob_to_regex(k, match_mode) for k in exclude_any if str(k).strip()]
            exclude_patterns = [p for p in exclude_patterns if p]
            if exclude_patterns:
                # exclude_or = "(" + "|".join(exclude_patterns) + ")"
                # exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

                
                exclude_or = "(?:" + "|".join(exclude_patterns) + ")"
                exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

            else:
                exclude_mask = pd.Series(False, index=work.index)
        else:
            exclude_mask = pd.Series(False, index=work.index)

        mask = include_mask & (~exclude_mask)

        group_idx = work.index[unassigned & mask]
        if len(group_idx) == 0:
            continue

        group_df = work.loc[group_idx]

        freq_sum = group_df[freq_col].sum()

        all_partnums = list(chain.from_iterable(group_df[partnums_col].tolist()))
        if dedupe_partnums:
            all_partnums = _dedupe_preserve_order(all_partnums)

        combined_parts = _dedupe_preserve_order(group_df[part_col].fillna("").astype(str).tolist())

        matched_rows_out.append({
            part_col: canonical_name,  # <-- group key goes into Part column
            freq_col: float(freq_sum),
            partnums_col: all_partnums,
            combined_parts_col: combined_parts,
            matched_keywords_col: {
                "include_any": include_any,
                "exclude_any": exclude_any
            }
        })

        unassigned.loc[group_idx] = False

    matched_df = pd.DataFrame(
        matched_rows_out,
        columns=[part_col, freq_col, partnums_col, combined_parts_col, matched_keywords_col]
    )


    remaining_df = work.loc[unassigned, [part_col, freq_col, partnums_col]].copy()
    remaining_df[combined_parts_col] = remaining_df[part_col].fillna("").astype(str).apply(lambda x: [x])
    remaining_df[matched_keywords_col] = [{"include_any": [], "exclude_any": []}] * len(remaining_df)

    final_df = pd.concat([matched_df, remaining_df], ignore_index=True)
    final_df[freq_col] = final_df[freq_col].round(2)
    final_df = final_df.sort_values(by=freq_col, ascending=False, kind="mergesort").reset_index(drop=True)

    display(final_df)

    return final_df


Analysis for Ford Site 130

In [307]:

def add_common_partnumbers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame, finds columns whose names contain 'PartNumbers',
    converts their stringified lists into real Python lists, and for each row:
      - finds the common elements across all these lists
      - stores them as:
            None  if there is no common element
            a set of strings if there is at least one common element
    in a new column 'common_partNumbers'.

    Returns a new DataFrame with the additional column.
    """
    df = df.copy()

    # 1) Identify all columns that contain 'PartNumbers' in their name
    partnumber_cols = [col for col in df.columns if "PartNumbers" in col]

    if not partnumber_cols:
        # No relevant columns – return df unchanged
        return df

    # 2) Helper to convert a value into a Python list
    def to_list(val):
        # Already a list → return as is
        if isinstance(val, list):
            return val

        # Missing / NaN → treat as empty list
        if pd.isna(val):
            return []

        # Strings that represent lists or comma-separated values
        if isinstance(val, str):
            s = val.strip()
            if not s:
                return []

            # Try to safely parse as Python literal first: "['123', '456']" or "[123, 456]"
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return list(parsed)
                # Scalar after eval → wrap in list
                return [parsed]
            except (ValueError, SyntaxError):
                # Fallback: treat as comma-separated string
                return [item.strip() for item in s.split(",") if item.strip()]

        # Any other scalar → wrap in list
        return [val]

    # 3) Function to compute common part numbers for a single row
    def get_common_parts(row):
        sets = []

        for col in partnumber_cols:
            lst = to_list(row[col])
            if lst:
                # Normalize everything to string so comparisons are consistent
                sets.append(set(map(str, lst)))

        if not sets:
            # No valid lists in this row → no common part numbers
            return None

        # Intersection of all sets
        common = set.intersection(*sets)

        # 🔑 Here's the behavior you requested:
        if not common:
            # empty set → store None
            return None
        else:
            # non-empty set → keep the set (could change to list if desired)
            return common

    # 4) Apply row-wise and create new column
    df["common_partNumbers"] = df.apply(get_common_parts, axis=1)

    return df


In [305]:



def compare_site_results_side_by_side(df_130, df_172, perc_thresh = 10):

    df_130 = combine_parts(df_130)
    df_172 = combine_parts(df_172)
    # create subset of parts with frequency above the threshold for both sites
    select_top_n_130 = df_130[df_130["frequency_%"].astype(int) >= perc_thresh]
    select_top_n_172 = df_172[df_172["frequency_%"].astype(int) >= perc_thresh]

    # determine subset size based on the smaller of the two sets to ensure a fair comparison
    subset_size = len(select_top_n_130) if len(select_top_n_130) > len(select_top_n_172) else len(select_top_n_172) 

    # merge the two dataframes on the "Part" column, keeping all parts from both sites (outer join)
    merged_resutls = df_130.merge(df_172, on="Part", how='outer')
    result_with_select_columns = merged_resutls[["Part", "frequency_%_x","frequency_%_y", "PartNumbers_x", "PartNumbers_y"]]

    result_with_select_columns.columns = ["Part", "Freq_130","Freq_172", "PartNumbers_130", "PartNumbers_172"]

    result_with_select_columns = add_common_partnumbers(result_with_select_columns)
    # sort the results by frequency in site 130 and select the top N parts based on the subset size
    result_with_select_columns = result_with_select_columns.sort_values("Freq_130", ascending=False).reset_index(drop=True)
    result_with_select_columns = result_with_select_columns.head(subset_size)
    
    return result_with_select_columns



def combine_parts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Combine rows that share the same PartNumbers by:
      - summing frequency_%,
      - selecting longest Part string as the main Part,
      - concatenating all unique Part values into combined_parts,
      - keeping all other original columns,
      - returning df with no repeated PartNumbers,
      - sorted by frequency_% descending.
    """
    df = df.copy()

    # 1) Create hashable group key for PartNumbers
    df["_group_key"] = df["PartNumbers"].apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    )

    # 2) Ensure frequency_% is numeric
    df["frequency_%"] = df["frequency_%"].astype(float)

    # 3) Compute combined frequency
    df["_combined_freq"] = df.groupby("_group_key")["frequency_%"].transform("sum")

    # 4) Combined unique part names (joined string)
    df["combined_parts"] = df.groupby("_group_key")["Part"].transform(
        lambda s: ", ".join(sorted(set(map(str, s))))
    )

    # 5) Determine longest part string for each group
    longest_part_map = (
        df.groupby("_group_key")["Part"]
        .apply(lambda s: max(s, key=lambda x: len(str(x))))
    )
    df["_longest_part"] = df["_group_key"].map(longest_part_map)

    # 6) Drop duplicates (keep first row)
    df = df.drop_duplicates(subset="_group_key", keep="first")

    # 7) Replace Part and frequency_% with aggregated versions
    df["Part"] = df["_longest_part"]
    df["frequency_%"] = df["_combined_freq"]

    # 8) Cleanup helper columns
    df = df.drop(columns=["_combined_freq", "_group_key", "_longest_part"])

    # 9) Sort final df
    df = df.sort_values("frequency_%", ascending=False).reset_index(drop=True)

    return df



def execute_model(request_tbl_df, search_key_words, labour_line_df, parts_line_df,
                  similarity_threshold, ignore_words, groups):

    # Normalize input: allow "Water Pump" or [["water","pump"], []]
    if isinstance(search_key_words, str):
        search_key_words = [[search_key_words], []]

    requests_filtered = filter_rows_by_keywords(
        request_tbl_df, "fldDescription", search_key_words, return_print = False
    )

    # Vectorized: filter parts only once
    req_ids = requests_filtered["fldId"].unique()
    filtered_parts_df = parts_line_df[parts_line_df["fldRequestRef"].isin(req_ids)].copy()
    parts_list = parts_summary(filtered_parts_df)

    
    # print("-----------------------------------------------------------------------------")
    final_df = combine_parts_by_keyword_groups(
        df=parts_list,
        groups=groups,
        part_col="Part",
        freq_col="frequency_%",       # <-- use your actual frequency column name
        partnums_col="PartNumbers",
        match_mode="contains"         # or "word"
    ).reset_index(drop=True)
    


    # display(final_df)
    return final_df



Illustration for Site 172

In [25]:


# db_server_172 = "DB_server_172"
# # key_wrd = "Water Pump Replace"

# server_conn_db_172 = SERVER_conn(db_server_172)

# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = pull_data_by_server(server_conn_db_172)
# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = clean_data(RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172)



In [26]:


# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = pull_data_by_server(server_conn_db_130)
# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = clean_data(RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130)

 

In [27]:

# # Repair 1: Water pump replace - Site 172

# # search_key_words_water_pump_172 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_172 = [["water", "pump"], []]

# print("Water pump - Site 172")
# resutlts_water_pump_site_172 = execute_model(request_tbl_db_172, search_key_words_water_pump_172, labourline_tbl_db_172, partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - ", "kit", "ASY", "Rep"])

# # Repair 1: Water pump replace - Site 130

# # search_key_words_water_pump_130 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_130 = [["water", "pump"], []]
# print("Water pump - Site 130")
# resutlts_water_pump_site_130 = execute_model(request_tbl_db_130, search_key_words_water_pump_130, labourline_tbl_db_130, partslines_tbl_db_130, similarity_threshold= 0.6, ignore_words=[" - ", "kit", "ASY", "Rep"])


# # Repair 2 : Catalytic Converter Replace - Site 172

# search_key_words_catalytic_replace_172 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_172, "fldDescription", search_key_words_battery_replace_172, False)

# print("Catalytic Converter - Site 172")
# resutlts_Catalytic_Converter_site_172 = execute_model(request_tbl_df= request_tbl_db_172, search_key_words= search_key_words_catalytic_replace_172, labour_line_df= labourline_tbl_db_172, parts_line_df= partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])
 

# # Repair 2 : Catalytic Converter Replace - Site 130

# search_key_words_catalytic_replace_130 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_130, "fldDescription", search_key_words_battery_replace_130, False)

# print("Catalytic Converter - Site 130")
# resutlts_Catalytic_Converter_site_130 = execute_model(request_tbl_df= request_tbl_db_130, search_key_words= search_key_words_catalytic_replace_130, labour_line_df= labourline_tbl_db_130, parts_line_df= partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "])
 


# # Repair 3 : Power Steering - Site 172

# # search_key_words_catalytic_replace_172 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_172 = [["Power steering"], []]
# print("Power Steering - Site 172")
# resutlts_Power_Steering_site_172 = execute_model(request_tbl_df = request_tbl_db_172, search_key_words = search_key_words_catalytic_replace_172, labour_line_df = labourline_tbl_db_172, parts_line_df = partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])


 
# # Repair 3 : Power Steering - Site 130

# # search_key_words_catalytic_replace_130 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_130 = [["Power steering"], []]
# print("Power Steering - Site 130")
# resutlts_Power_Steering_site_130 = execute_model(request_tbl_df = request_tbl_db_130, search_key_words = search_key_words_catalytic_replace_130, labour_line_df = labourline_tbl_db_130, parts_line_df = partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "]) 
 

In [28]:

# compare_water_pump = resutlts_water_pump_site_172.merge(resutlts_water_pump_site_130, on="Part")
# compare_water_pump


# # print(type(resutlts_water_pump_site_172))

In [29]:
# Construct queries 


def is_validModel(server_conn, model):
    query = f" SELECT * FROM Veh_tblModel WHERE fldName = '{model}' AND fldInActive = 0"

    retults = db_request(query, server_conn)
    return len(retults)

def query_constructor(model):

    Queries= dict()

    RO_all = f"SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_requests = f"SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
    query_all_PartsLine = f"SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded, PC.fldPartDescription as PC_PartDesc, PC.fldPartsMasterRef FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId INNER JOIN Parts_tblPartsCurrent PC WITH(NOLOCK) on PL.fldPartNumber = PC.fldPartNumber WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    Queries["RO_tbl"] = RO_all
    Queries["Req_tbl"] = query_all_requests
    Queries["Parts_tbl"] = query_all_PartsLine
    Queries["Labour_tbl"] = query_all_LabourLine

    return Queries

In [30]:


def data_pull(modelName,db_server):
    
    # key_wrd = "Replace Water Pump"
    server_conn = SERVER_conn(db_server)
    
    if not is_validModel(server_conn, modelName):
        raise ValueError("Provided Model Name does not exists")
    queries = query_constructor(modelName)
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = pull_data_by_server_with_args(server_conn, queries, modelName)
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl)
    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl


def save_data(directoryPath : str, df, siteName):
    # df.to_csv('dat/site')
    return 


def run_model(key_wrd, request_tbl, labourline_tbl, partslines_tbl):
    execute_model(request_tbl_df = request_tbl, search_key_words = key_wrd, labour_line_df = labourline_tbl, parts_line_df = partslines_tbl, similarity_threshold=0.7, ignore_words=[" - "]) 




    

In [31]:
def dataset_info(RO_tbl, req_tbl, labour_tbl, parts_tbl, site):

    print(f"Sample Size: Site {site}")

    print(f"RO_tbl: {len(RO_tbl)}")
    print(f"Req_tbl: {len(req_tbl)}")
    print(f"LabourLines_tbl: {len(labour_tbl)}")
    print(f"PartLines_tbl: {len(parts_tbl)}")

In [33]:
# modelName = "F-150"
# Servers = ["DB_server_130",'DB_server_172']

# RO_tbl_130_f150, request_tbl_130_f150, labourline_tbl_130_f150, partslines_tbl_130_f150 = data_pull(modelName, Servers[0])

In [34]:
# partslines_tbl_130_f150.head()

In [122]:

# WATER PUMP", "PUMP ASY", "PUMP ASY - WA*", "KIT - WATER"
#  Keywords for grouping parts into categories (example provided, can be expanded as needed)
groups= {
    "WATER PUMP (KIT/ASY)": [["WATER", "PUMP"], ["Gasket","Pulley","HOSE","COVER","OIL","CONNECTION","WATER BYP","FUE","TUBE","ADAPTOR","WASHER"]],
    "MC YELLOW COOLANT":[["COOLANT"], []],
    "GASKET - WATER PUMP ":[["GASKET"],[]],
    "POWER STEERING (ALL)": [["POWER STEERING", "P/S", "STEERING PUMP"],[]],
    "CATALYTIC CONVERTER (ALL)": [["CATALYTIC", "CAT CONVERTER", "CONVERTER"],[]]
}

In [36]:
# key_wrd = "Water Pump"
# results_tbl_130_f150 = execute_model(request_tbl_df = request_tbl_130_f150, search_key_words = key_wrd, labour_line_df = labourline_tbl_130_f150, parts_line_df = partslines_tbl_130_f150, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

In [37]:

def save_data_locally(data, base_path, server_name):
    # Build final directory: base_path/server_name
    final_path = os.path.join(base_path, server_name)
    
    # Create directory if it doesn't exist
    os.makedirs(final_path, exist_ok=True)
    
    # Basic validation
    if not isinstance(data, dict):
        raise ValueError("`data` must be a dict of pandas DataFrames.")
    
    # Save each DataFrame
    for key, df in data.items():
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Value for key '{key}' is {type(df)}, expected pandas.DataFrame")

        file_path = os.path.join(final_path, f"{key}.csv")
        print(f"Saving {key} -> {file_path}")  # DEBUG print
        df.to_csv(file_path, index=False)

    print(f"Done. Saved {len(data)} file(s) into: {final_path}")


# # ---- TEST IT ----
# data = {
#     "RO_tbl": pd.DataFrame({"a": [1, 2], "b": [3, 4]}),
#     "Req_tbl": pd.DataFrame({"x": [10, 20]}),
#     "Parts_tbl": pd.DataFrame({"part": ["p1", "p2"]}),
# }

# # Use a path you KNOW exists & can write to
# base_path = "./data"   # <-- safer than "/data/" on many systems

# save_data_locally(data, base_path)

In [38]:
# # this is the main execution block where we pull data for both sites, run analysis, and save results locally. You can modify the modelName and Servers list as needed. 

# modelName = "Escape"
# Servers = ["DB_server_130",'DB_server_172']

# RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_pull(modelName, Servers[0])
# RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_pull(modelName, Servers[1])



# dataset_info(RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130, "130")
# print("---------------------------------------------------------------------")
# dataset_info(RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172, "172")


# data = {"server_130": {
#                 "RO_tbl": RO_tbl_130,
#                 "Req_tbl": request_tbl_130, 
#                 "Labor_tbl":labourline_tbl_130, 
#                 "Parts_tbl": partslines_tbl_130},
#         "server_172": {
#                 "RO_tbl": RO_tbl_172,
#                 "Req_tbl": request_tbl_172, 
#                 "Labor_tbl":labourline_tbl_172, 
#                 "Parts_tbl": partslines_tbl_172}
# }
# base_path = "./data"   # the base dir for data

# for key, data in data.items():
#      save_data_locally(data, base_path, key)



In [ ]:

def data_loading_from_local(base_path, server_name):
    # full_path = os.path.join(base_path, server_name)

    RO_file_name = os.path.join(base_path, server_name, "RO_tbl.csv")
    Req_file_name = os.path.join(base_path, server_name, "Req_tbl.csv")
    Labor_file_name = os.path.join(base_path, server_name, "Labor_tbl.csv")
    Parts_file_name = os.path.join(base_path, server_name, "Parts_tbl.csv")

    RO_df = pd.read_csv(RO_file_name)
    Req_df = pd.read_csv(Req_file_name)
    Labor_df = pd.read_csv(Labor_file_name)
    Parts_df = pd.read_csv(Parts_file_name)

    return RO_df, Req_df, Labor_df, Parts_df
 

In [94]:
# loading data sets
base_path = "./data"

RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_loading_from_local(base_path, "server_130")
RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_loading_from_local(base_path, "server_172")


In [98]:
def refactor_description_column(df, description_col="fldPartDesc"):
    
    # this function will select all the rows where column fldPartMasterRef = 1 and then for those rows, it will replace the value in fldDescription column with the value in fldPartDesc column. However, the value in fldPartDesc column should be not empty, otherwise the value in fldDescription column will remain unchanged. Before replacing the value, make a copy of the original value in fldDescription column and store it in a new column called Original_fldDescription. aslo, add a column named changed_description which will be True if the description is changed and False if it is not changed.


    mask = (df["fldPartsMasterRef"] == 1) & (df["fldPartDesc"].notna()) & (df["fldPartDesc"] != "")
    df["Original_" + description_col] = df[description_col]
    df["changed_description"] = False
    df.loc[mask, description_col] = df.loc[mask, "fldPartDesc"]
    df.loc[mask, "changed_description"] = True
    return df

In [96]:
partslines_tbl_130.columns

Index(['fldID', 'fldRequestRef', 'fldSequence', 'fldPartNumber', 'fldPartDesc',
       'fldRequested', 'fldShipped', 'fldOrderType', 'fldDateAdded',
       'PC_PartDesc', 'fldPartsMasterRef'],
      dtype='object')

In [99]:

partslines_tbl_130 = refactor_description_column(partslines_tbl_130)
partslines_tbl_172 = refactor_description_column(partslines_tbl_172)
 

In [100]:

# Next work on adding the new description to the model
# Implement the logic to pick the correct description



In [119]:
# partslines_tbl_130["changed_description"].value_counts()

In [309]:

key_wrds = ["Water Pump", "Power steering", "Catalytic Converter"]

key_wrd = key_wrds[0]

results130_water_pump = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results172_water_pump = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)



,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,WATER PUMP (KIT/ASY),95.69,"[PW 556, PW 625, PW 579, PW 545, PW 493, PW 68...","[PUMP ASY - WATER, PUMP ASY - WAT, WATER PUMO,...","{'include_any': ['WATER', 'PUMP'], 'exclude_an..."
1,GASKET - WATER PUMP,79.14,"[BE8Z 8507 A, 9L8Z 8507 A, 1S7Z 8507 AE, AM5Z ...","[GASKET - WATER PUMP, GASKET - WATER, GASKET, ...","{'include_any': ['GASKET'], 'exclude_any': []}"
2,MC YELLOW COOLANT,46.76,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2]","[MC YELLOW COOLANT 4L (PREMIX), MC YELLOW COOL...","{'include_any': ['COOLANT'], 'exclude_any': []}"
3,NUT,33.09,"[W520214 S440, W520415 S442, W715135 S440]",[NUT],"{'include_any': [], 'exclude_any': []}"
4,ANTI-FREEZE,30.94,"[CVC 3 B2, CVC 13 G, CVC 7 B2, CVC 3 D1LB, CVC...",[ANTI-FREEZE],"{'include_any': [], 'exclude_any': []}"
...,...,...,...,...,...
59,HOSE - RADIATO,0.72,[KM 5130],[HOSE - RADIATO],"{'include_any': [], 'exclude_any': []}"
60,KIT - TENSION,0.72,[YS 354],[KIT - TENSION],"{'include_any': [], 'exclude_any': []}"
61,KIT - TENSION PULLEY,0.72,[YS 354],[KIT - TENSION PULLEY],"{'include_any': [], 'exclude_any': []}"
62,LUBE KIT,0.72,[PKFL-500],[LUBE KIT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,WATER PUMP (KIT/ASY),100.00,"[DS7Z 8501 E, 7S7Z 8501 C, 4S4Z 8501 AA, 4S4Z ...","[PUMP ASY - WATER, PUILL PUMP PLEASE]","{'include_any': ['WATER', 'PUMP'], 'exclude_an..."
1,GASKET - WATER PUMP,85.72,"[BE8Z 8507 A, BM5Z 6584 B]","[GASKET - WATER PUMP, GASKET - VALVE ROCKER AR...","{'include_any': ['GASKET'], 'exclude_any': []}"
2,ANTI-FREEZE,24.68,"[VC 13 G, VC 3 B, VC 7 B]",[ANTI-FREEZE],"{'include_any': [], 'exclude_any': []}"
3,V-BELT,7.79,"[JK6 617 A, F1EZ 8620 A, JK6 867 A, JK3 211 A]",[V-BELT],"{'include_any': [], 'exclude_any': []}"
4,NUT,6.49,"[W715618 S437, W520214 S440, W520415 S442, W52...",[NUT],"{'include_any': [], 'exclude_any': []}"
5,NUT - LOCKING,5.19,[W520102 S442],[NUT - LOCKING],"{'include_any': [], 'exclude_any': []}"
6,RETAINER - BEARING,5.19,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"
7,SCREW AND WASHER ASY,5.19,[W716136 S442],[SCREW AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"
8,FILTER ASY - OIL,3.90,[BE8Z 6731 AC],[FILTER ASY - OIL],"{'include_any': [], 'exclude_any': []}"
9,BOLT,3.90,"[7N5Z 00812 A, W712146 S437, 5F9Z 4682 AA, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"


In [322]:

compare_site_results_side_by_side(results130_water_pump, results172_water_pump, perc_thresh=10)
 

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172,common_partNumbers
0,WATER PUMP (KIT/ASY),95.69,100.00,"[PW 556, PW 625, PW 579, PW 545, PW 493, PW 68...","[DS7Z 8501 E, 7S7Z 8501 C, 4S4Z 8501 AA, 4S4Z ...",None
1,GASKET - WATER PUMP,79.14,85.72,"[BE8Z 8507 A, 9L8Z 8507 A, 1S7Z 8507 AE, AM5Z ...","[BE8Z 8507 A, BM5Z 6584 B]","{BE8Z 8507 A, BM5Z 6584 B}"
2,MC YELLOW COOLANT,46.76,1.30,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2]",[nan],None
3,SCREW AND WASHER ASY,33.10,5.19,[W716136 S442],[W716136 S442],{W716136 S442}
4,NUT,33.09,6.49,"[W520214 S440, W520415 S442, W715135 S440]","[W715618 S437, W520214 S440, W520415 S442, W52...","{W520214 S440, W520415 S442}"
5,ANTI-FREEZE,30.94,24.68,"[CVC 3 B2, CVC 13 G, CVC 7 B2, CVC 3 D1LB, CVC...","[VC 13 G, VC 3 B, VC 7 B]",None
6,V-BELT,28.78,7.79,"[JK6 441, JK3 211 A, JK6 612, JK6 617 A, ZL608...","[JK6 617 A, F1EZ 8620 A, JK6 867 A, JK3 211 A]","{JK6 617 A, JK3 211 A}"
7,RETAINER - BEARING,23.02,5.19,[YS4Z 3N324 AA],[YS4Z 3N324 AA],{YS4Z 3N324 AA}
8,NUT - LOCKING,22.30,5.19,[W520102 S442],[W520102 S442],{W520102 S442}
9,BOLT,21.58,3.90,"[W716075 S442, 5F9Z 4682 AA, W715681 S900, W50...","[7N5Z 00812 A, W712146 S437, 5F9Z 4682 AA, W71...","{BE8Z 6A340 A, W718250 S439, W716075 S442, 5F9..."


In [311]:
key_wrd = key_wrds[1]
results130_power_str = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_power_str = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,SENSOR - STEER,44.12,[CL8Z 3F818 A],[SENSOR - STEER],"{'include_any': [], 'exclude_any': []}"
1,BOLT,39.71,"[W712250 S437, W713065 S439]",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,BULK MERCON V,10.29,[CXT-5-L],[BULK MERCON V],"{'include_any': [], 'exclude_any': []}"
3,WATER PUMP (KIT/ASY),8.82,"[STP 152, STP 182, ZLCRD21-5271]","[PUMP ASY - POW, STEERING PUMP]","{'include_any': ['WATER', 'PUMP'], 'exclude_an..."
4,GEAR ASY - STE,5.88,"[STE 419, STE 175, STE 98, STE 282]",[GEAR ASY - STE],"{'include_any': [], 'exclude_any': []}"
5,V-BELT,4.41,"[JK6 844 C, JK6 931 AA, JK6 455 C]",[V-BELT],"{'include_any': [], 'exclude_any': []}"
6,SENSOR - STEERING RO,4.41,[CL8Z 3F818 A],[SENSOR - STEERING RO],"{'include_any': [], 'exclude_any': []}"
7,LINK,2.94,"[MEF 166, MEF 200]",[LINK],"{'include_any': [], 'exclude_any': []}"
8,SEAL,2.94,[388898 S],[SEAL],"{'include_any': [], 'exclude_any': []}"
9,LUBE KIT,2.94,[PKFL-500],[LUBE KIT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,SENSOR - STEERING ROTATION,38.76,[CL8Z 3F818 A],[SENSOR - STEERING ROTATION],"{'include_any': [], 'exclude_any': []}"
1,BOLT,33.33,"[7N5Z 00812 A, W714807 S900, W716075 S442, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,COLUMN ASY - STEERING,14.73,"[CL8Z 3C529 D, PZ1Z 3C529 AQ, CL8Z 3C529 B]",[COLUMN ASY - STEERING],"{'include_any': [], 'exclude_any': []}"
3,GEAR ASY - STEERING,13.18,"[CV6Z 3504 WE, CV6Z 3504 AGE, CV6Z 3504 EE, CV...",[GEAR ASY - STEERING],"{'include_any': [], 'exclude_any': []}"
4,BOLT - HEX.HEAD,7.75,[W711137 S442],[BOLT - HEX.HEAD],"{'include_any': [], 'exclude_any': []}"
5,NUT - HEX.,7.75,"[W520203 S442, W520215 S440, W520113 S442, W70...",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
6,NPN PART UCS HISTORY,6.98,[NPN],[NPN PART UCS HISTORY],"{'include_any': [], 'exclude_any': []}"
7,COLUMN ASY - STEERIN,6.98,[CL8Z 3C529 C],[COLUMN ASY - STEERIN],"{'include_any': [], 'exclude_any': []}"
8,NUT,6.20,"[W520214 S443, W715135 S440, W520415 S442]",[NUT],"{'include_any': [], 'exclude_any': []}"
9,ALTERNATOR ASY,4.65,"[CJ5Z 10346 D, CJ5Z 10346 C, CJ5Z 10346 A, CJ5...",[ALTERNATOR ASY],"{'include_any': [], 'exclude_any': []}"


In [312]:

compare_site_results_side_by_side(results130_power_str, results172_power_str)
 

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172,common_partNumbers
0,STEERING TORQUE SENSOR,51.47,NaN,[CL8Z 3F818 A],NaN,{CL8Z 3F818 A}
1,BOLT,39.71,33.33,"[W712250 S437, W713065 S439]","[7N5Z 00812 A, W714807 S900, W716075 S442, W71...","{W713065 S439, W712250 S437}"
2,BULK MERCON V,10.29,NaN,[CXT-5-L],NaN,{CXT-5-L}
3,WATER PUMP (KIT/ASY),8.82,NaN,"[STP 152, STP 182, ZLCRD21-5271]",NaN,"{STP 182, STP 152, ZLCRD21-5271}"


In [313]:
key_wrd = key_wrds[2]
results130_catal = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_catal = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,CATALYTIC CONVERTER (ALL),81.82,"[CV6Z 5E212 D, CV6Z 5E212 F, LX6Z 5E212 KZ, JJ...","[CONVERTER ASY, REAR CONVERTER]","{'include_any': ['CATALYTIC', 'CAT CONVERTER',..."
1,BOLT,63.64,"[5F9Z 4682 AA, W500635 S439, W716075 S442, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,NUT - HEX.,63.64,"[W520103 S403, W520203 S442, W520103 S442]",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
3,GASKET - WATER PUMP,45.45,"[BB5Z 6L612 A, CV6Z 9450 D, CV6Z 9450 E, AM5Z ...","[GASKET, GASKET - EXHAUST MAN]","{'include_any': ['GASKET'], 'exclude_any': []}"
4,CLAMP - EXHAUST,45.45,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",[CLAMP - EXHAUST],"{'include_any': [], 'exclude_any': []}"
5,BOLT AND WASHER ASY,18.18,"[W711806 S442, W709601 S442]",[BOLT AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"
6,NUT,18.18,"[W520415 S442, W716271 S437]",[NUT],"{'include_any': [], 'exclude_any': []}"
7,SEAL - REFER TO (PK-CN1Z) **,18.18,[CN1Z 7H424 B],[SEAL - REFER TO (PK-CN1Z) **],"{'include_any': [], 'exclude_any': []}"
8,SEAL,18.18,[CV6Z 7086 B],[SEAL],"{'include_any': [], 'exclude_any': []}"
9,RETAINER - BEARING,18.18,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords


In [314]:
compare_site_results_side_by_side(results130_catal, results172_catal)

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172,common_partNumbers
0,CATALYTIC CONVERTER (ALL),81.82,NaN,"[CV6Z 5E212 D, CV6Z 5E212 F, LX6Z 5E212 KZ, JJ...",NaN,"{ZL16410, CV6Z 5E212 D, JJ5Z 5E212 B, CV6Z 5E2..."
1,BOLT,63.64,NaN,"[5F9Z 4682 AA, W500635 S439, W716075 S442, W71...",NaN,"{W716075 S442, W500233 S442, W718250 S439, W50..."
2,NUT - HEX.,63.64,NaN,"[W520103 S403, W520203 S442, W520103 S442]",NaN,"{W520103 S442, W520203 S442, W520103 S403}"
3,CLAMP - EXHAUST,45.45,NaN,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",NaN,"{JX6Z 5A215 C, LX6Z 5A215 A, LX6Z 5A215 D}"
4,GASKET - WATER PUMP,45.45,NaN,"[BB5Z 6L612 A, CV6Z 9450 D, CV6Z 9450 E, AM5Z ...",NaN,"{CV6Z 9450 D, CV6Z 9450 E, ZL31578, AM5Z 9450 ..."
5,BOLT AND WASHER ASY,18.18,NaN,"[W711806 S442, W709601 S442]",NaN,"{W709601 S442, W711806 S442}"
6,NUT,18.18,NaN,"[W520415 S442, W716271 S437]",NaN,"{W716271 S437, W520415 S442}"
7,SEAL - REFER TO (PK-CN1Z) **,18.18,NaN,[CN1Z 7H424 B],NaN,{CN1Z 7H424 B}
8,SEAL,18.18,NaN,[CV6Z 7086 B],NaN,{CV6Z 7086 B}
9,RETAINER - BEARING,18.18,NaN,[YS4Z 3N324 AA],NaN,{YS4Z 3N324 AA}


In [315]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
water_pump_keywrds = ["Water|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER"] 


In [316]:

all_parts_dfs_combined = results172_catal.copy()

all_parts_dfs_combined = pd.concat(
    [partslines_tbl_172, partslines_tbl_130])


In [317]:
def group_dfs(*dfs):
    return pd.concat(dfs, ignore_index=True)


In [318]:

def search_parts_by_keyword(df, search__keys):
    result = df[(df["fldPartDesc"].str.contains(search__keys[0], case= False, na=False)) &
                   (~df["fldPartDesc"].str.contains(search__keys[1], case= False, na=False))]
    display(result["fldPartDesc"].unique().tolist())
 

In [319]:
# "gasket", "coolant", "Nut", "Anti|Freeze",
key_words_for_water_pump_parts = ["belt", "nut", "bolt", "screw", "retainer", "seal", "oil", "stud"]


# Nuts, Seal, Srew, Retainer, stud

# key_words_for_catal_parts = ["converter", "bolt", 'nut', "gasket", "clamp", "washer", "seal", "retainer", "coolant", "hanger", "tube", "sensor", "insulator"]

for part_key_wrd in key_words_for_water_pump_parts:
    display(results172_water_pump[results172_water_pump["Part"].str.contains(part_key_wrd, case=False)])

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
3,V-BELT,7.79,"[JK6 617 A, F1EZ 8620 A, JK6 867 A, JK3 211 A]",[V-BELT],"{'include_any': [], 'exclude_any': []}"
12,BELT - TIMING,2.60,[BE8Z 6268 C],[BELT - TIMING],"{'include_any': [], 'exclude_any': []}"
21,TENSIONER - TIMING BELT,1.30,[BM5Z 6K254 A],[TENSIONER - TIMING BELT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
4,NUT,6.49,"[W715618 S437, W520214 S440, W520415 S442, W52...",[NUT],"{'include_any': [], 'exclude_any': []}"
5,NUT - LOCKING,5.19,[W520102 S442],[NUT - LOCKING],"{'include_any': [], 'exclude_any': []}"
13,RETAINER - NUT,2.60,[CCPZ 3B477 G],[RETAINER - NUT],"{'include_any': [], 'exclude_any': []}"
26,NUT - HEX.,1.30,[W520203 S442],[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
9,BOLT,3.9,"[7N5Z 00812 A, W712146 S437, 5F9Z 4682 AA, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
15,BOLT - HEX.HEAD,1.3,[BE8Z 6379 AB],[BOLT - HEX.HEAD],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
7,SCREW AND WASHER ASY,5.19,[W716136 S442],[SCREW AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
6,RETAINER - BEARING,5.19,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"
13,RETAINER - NUT,2.60,[CCPZ 3B477 G],[RETAINER - NUT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
17,SEAL,1.3,[9L8Z 1177 C],[SEAL],"{'include_any': [], 'exclude_any': []}"
18,SEALANT - SILICONE,1.3,[TA 357],[SEALANT - SILICONE],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
8,FILTER ASY - OIL,3.9,[BE8Z 6731 AC],[FILTER ASY - OIL],"{'include_any': [], 'exclude_any': []}"
19,SEPARATOR ASY - OIL,1.3,[DS7Z 6A785 C],[SEPARATOR ASY - OIL],"{'include_any': [], 'exclude_any': []}"
23,OIL - AUTOMATIC TRANSMISSION,1.3,[XT 10 QLVC],[OIL - AUTOMATIC TRANSMISSION],"{'include_any': [], 'exclude_any': []}"
24,OIL COOLER ASY,1.3,[DS7Z 6B856 A],[OIL COOLER ASY],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords


In [320]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
# water_pump_keywrds = ["Water Pump|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER|Coupling|Steering|coolant"] 

water_pump_gasket_keywrds = ["gasket - water", "None"] 

# water_pump_keywrds
combo_df = group_dfs(partslines_tbl_130, partslines_tbl_172)

search_parts_by_keyword(combo_df, water_pump_gasket_keywrds)


['GASKET - WATER', 'GASKET - WATER PUMP']

In [321]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
power_steering_keywrds = ["steering", "pump"] 

search_parts_by_keyword(combo_df, power_steering_keywrds)


['SENSOR - STEERING RO',
 'WHEEL ASY - STEERING',
 'STEERING TORQUE SENSOR',
 'LOCK ASY - STEERING',
 'GEAR ASY - STEERING',
 'used steering column',
 'SWITCH ASY - STEERING WHEEL',
 'CORE - GEAR ASY - STEERING',
 'COLUMN ASY - STEERING',
 'P&A STEERING SHAFT BOLT',
 'SENSOR ASY - STEERING ROTATION',
 'LOCK ASY - STEERING AND IGNITI',
 'SENSOR - STEERING ROTATION']